# TinyHybridNet + QAT: Quantization-Aware Training on Tiny ImageNet

## Overview
This notebook demonstrates **quantization-aware training (QAT)** on a hybrid efficient CNN architecture.

### Architecture Highlights
- **FireMobileResidual blocks**: Combine squeeze-excitation, depthwise separable convolutions, and residual connections
- **Mixed convolution strategies**: 1×1 bottlenecks + 3×3 depthwise layers
- **Efficient design**: Suitable for deployment with low-precision (INT8) quantization

### Objectives
1. Train a hybrid CNN on Tiny ImageNet (64×64 images, 200 classes)
2. Quantize model weights and activations to INT8
3. Measure accuracy drop and model size compression
4. Track metrics throughout training and quantization

### Dataset
- **Tiny ImageNet-200**: 64×64 RGB images, 200 object classes
- **Training**: 90% of data (~82K images), **Validation**: 10% (~9K images)

### Expected Outputs
- FP32 baseline model with best validation accuracy
- INT8 quantized model via QAT
- Side-by-side accuracy and size comparison


## Section 1: Configuration & Environment

In [1]:
# CELL 1: IMPORTS & ENVIRONMENT SETUP

import os
import sys
import time
import random
import copy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.ao.quantization as tq
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm

# Import shared utilities
from ml_utils import (
    set_seed, get_system_info, model_summary,
    train_epoch, evaluate, save_model, load_model,
    ExperimentConfig, create_results_summary, count_parameters
)

import kagglehub

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

print("✓ All imports successful")

/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports successful


In [2]:
# CELL 2: EXPERIMENT CONFIGURATION

# Random seed for reproducibility
SEED = 42
set_seed(SEED)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.backends.quantized.engine = "fbgemm"

# System information
system_info = get_system_info()
print("System Information:")
for key, value in system_info.items():
    print(f"  {key}: {value}")

# Model & Training Configuration
config = ExperimentConfig({
    # Reproducibility
    "seed": SEED,
    "device": str(device),

    # Dataset
    "dataset": "tiny-imagenet-200",
    "img_size": 64,
    "num_classes": 200,
    "train_val_split": 0.9,  # 90% train, 10% val

    # Data loading
    "batch_size": 64,
    "num_workers": 0,
    "pin_memory": True,

    # FP32 Training
    "fp32_epochs": 2,
    "fp32_lr": 3e-4,
    "fp32_weight_decay": 5e-4,
    "fp32_label_smoothing": 0.1,

    # QAT Fine-tuning
    "qat_epochs": 2,
    "qat_lr": 1e-4,
    "qat_weight_decay": 5e-4,
    "qat_freeze_bn_epoch": 3,          # epoch (0-indexed) after which BN stats are frozen
    "qat_disable_observer_epoch": 3,   # epoch after which fake-quant observers are frozen

    # Paths
    "save_dir": "./saved_models_hybrid_qat",
    "checkpoint_dir": "./saved_models_hybrid_qat/checkpoints",
})

System Information:
  device: cuda
  torch_version: 2.5.1+cu121
  cuda_available: True
  cuda_version: 12.1
  gpu_count: 1
  gpu_name: NVIDIA GeForce RTX 4060 Laptop GPU
  gpu_memory_gb: 8.186822656


## Section 2: Dataset & Data Loading

**Preprocessing Strategy:**
- **Training**: RandomResizedCrop (0.6-1.0 scale), HorizontalFlip, ColorJitter, Normalization
- **Validation**: Resize, CenterCrop, Normalization (ImageNet stats)
- **Split**: 90/10 train/val with fixed seed for reproducibility


In [3]:
# CELL 3: DOWNLOAD & LOAD DATASET

# Download dataset
print("Downloading Tiny ImageNet-200...")
dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")

print(f"✓ Dataset available at: {train_path}")

# Transforms
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(config["img_size"], scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

transform_val = transforms.Compose([
    transforms.Resize(config["img_size"]),
    transforms.CenterCrop(config["img_size"]),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# Base dataset (only for size + class structure)
base_dataset = datasets.ImageFolder(train_path)

n_total = len(base_dataset)
n_train = int(config["train_val_split"] * n_total)

# Deterministic shuffle of indices
g = torch.Generator().manual_seed(SEED)
indices = torch.randperm(n_total, generator=g).tolist()

train_indices = indices[:n_train]
val_indices = indices[n_train:]

# Separate datasets with transforms
train_dataset_full = datasets.ImageFolder(train_path, transform=transform_train)
val_dataset_full = datasets.ImageFolder(train_path, transform=transform_val)

train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(val_dataset_full, val_indices)

# DataLoaders (safe config for Jupyter stability)
train_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=config["pin_memory"]
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=config["pin_memory"]
)

print("\n✓ Dataset loaded:")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")
print(f"  Total classes: {len(base_dataset.classes)}")
print(f"  Batch size: {config['batch_size']}")

os.makedirs(config["save_dir"], exist_ok=True)
os.makedirs(config["checkpoint_dir"], exist_ok=True)

# Save configuration
config.save(os.path.join(config["save_dir"], "config.json"))
print("\n✓ Configuration loaded and saved")
print(f"\nKey Settings:\n{config}")

✓ Dataset available at: /home/rafael/.cache/kagglehub/datasets/akash2sharma/tiny-imagenet/versions/1/tiny-imagenet-200/train

✓ Dataset loaded:
  Training samples: 90000
  Validation samples: 10000
  Total classes: 200
  Batch size: 64

✓ Configuration loaded and saved

Key Settings:
{
  "seed": 42,
  "device": "cuda",
  "dataset": "tiny-imagenet-200",
  "img_size": 64,
  "num_classes": 200,
  "train_val_split": 0.9,
  "batch_size": 64,
  "num_workers": 0,
  "pin_memory": true,
  "fp32_epochs": 2,
  "fp32_lr": 0.0003,
  "fp32_weight_decay": 0.0005,
  "fp32_label_smoothing": 0.1,
  "qat_epochs": 2,
  "qat_lr": 0.0001,
  "qat_weight_decay": 0.0005,
  "qat_freeze_bn_epoch": 3,
  "qat_disable_observer_epoch": 3,
  "save_dir": "./saved_models_hybrid_qat",
  "checkpoint_dir": "./saved_models_hybrid_qat/checkpoints"
}


## Section 3: Model Definition

### TinyHybridNet Architecture
- **Stem**: 3 → 32 channels (3×3 conv)
- **FireMobileResidual blocks**: Combine depthwise separable + bottleneck + residuals
  - 1×1 squeeze layer reduces channel count by 75%
  - 3×3 depthwise convolution for spatial processing
  - 1×1 expansion back to output channels
  - Skip connection with identity/projection layer
- **Output**: AdaptiveAvgPool → Linear classifier
- **Quantization stubs**: QuantStub/DeQuantStub for QAT support

### Design Rationale
- Depthwise separable layers reduce FLOPs (~9× vs standard conv for 3×3)
- Squeeze-excitation improves parameter efficiency
- Residual connections improve gradient flow and model expressiveness


In [4]:
# CELL 4: MODEL ARCHITECTURE

class FireMobileResidual(nn.Module):
    """
    Fire-inspired mobile residual block.

    Combines:
    - Squeeze: 1×1 conv reduces channels
    - Mobile: Depthwise separable conv processes spatially
    - Residual: Skip connection with learned projection

    Args:
        in_ch: Input channels
        out_ch: Output channels
        stride: Stride for depthwise conv (enables downsampling)
        squeeze_ratio: Fraction of output channels to use in squeeze layer (default 0.25)
    """

    def __init__(self, in_ch, out_ch, stride=1, squeeze_ratio=0.25):
        super().__init__()

        squeeze_ch = max(16, int(out_ch * squeeze_ratio))

        # Squeeze-Excitation inspired bottleneck
        self.block = nn.Sequential(
            # 1×1 bottleneck
            nn.Conv2d(in_ch, squeeze_ch, 1, bias=False),
            nn.BatchNorm2d(squeeze_ch),
            nn.ReLU(inplace=True),

            # 3×3 Depthwise convolution
            nn.Conv2d(
                squeeze_ch, squeeze_ch,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=squeeze_ch,  # Depthwise: each input channel has own filters
                bias=False
            ),
            nn.BatchNorm2d(squeeze_ch),
            nn.ReLU(inplace=True),

            # 1×1 expansion
            nn.Conv2d(squeeze_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch)
        )

        # Residual shortcut with projection if needed
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

        # Quantization-friendly addition (FloatFunctional supports fake-quant
        # insertion on the residual add during QAT/convert).
        self.skip_add = tq.FloatFunctional()

    def forward(self, x):
        out = self.block(x)
        identity = self.shortcut(x)
        out = self.skip_add.add(out, identity)
        return torch.relu(out)


class TinyHybridNet(nn.Module):
    """
    Efficient hybrid CNN for small images (64×64).

    Uses depthwise separable convolutions, squeeze bottlenecks, and residuals
    for parameter efficiency while maintaining expressiveness.
    Designed for INT8 quantization via QAT.

    Args:
        num_classes: Number of output classes (default 200)
    """

    def __init__(self, num_classes=200):
        super().__init__()

        # Quantization stubs for QAT
        self.quant = tq.QuantStub()
        self.dequant = tq.DeQuantStub()

        # Stem layer
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )

        # Feature extraction
        self.features = nn.Sequential(
            FireMobileResidual(32, 64),             # 64×64
            FireMobileResidual(64, 64),              # 64×64
            FireMobileResidual(64, 128, stride=2),   # 32×32
            FireMobileResidual(128, 128),            # 32×32
            FireMobileResidual(128, 256, stride=2),  # 16×16
            FireMobileResidual(256, 256),             # 16×16
        )

        # Classification head
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.quant(x)
        x = self.stem(x)
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        x = self.dequant(x)
        return x


# Create model
model = TinyHybridNet(num_classes=config["num_classes"]).to(device)

# Print model info
print("\n✓ TinyHybridNet created")
print(f"\n{model_summary(model, device)}")

AttributeError: module 'torch.ao.quantization' has no attribute 'FloatFunctional'

## Section 4: Training (FP32 Baseline)

### FP32 Training Strategy
- **Optimizer**: AdamW with learning rate decay
- **Scheduler**: Cosine annealing for smooth learning rate decay
- **Loss**: CrossEntropyLoss with label smoothing (0.1)
- **Epochs**: 30 epochs to ensure convergence
- **Checkpoint**: Save best model based on validation accuracy


In [ ]:
# CELL 5: FP32 BASELINE TRAINING

# Setup training
criterion = nn.CrossEntropyLoss(label_smoothing=config["fp32_label_smoothing"])
optimizer = optim.AdamW(
    model.parameters(),
    lr=config["fp32_lr"],
    weight_decay=config["fp32_weight_decay"]
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config["fp32_epochs"]
)

print("Starting FP32 training...")
print(f"Configuration:\n  Epochs: {config['fp32_epochs']}\n  LR: {config['fp32_lr']}\n  Weight Decay: {config['fp32_weight_decay']}")

fp32_results = {}
best_fp32_acc = 0.0

fp32_ckpt_path = os.path.join(config["save_dir"], "tinyhybrid_fp32.pth")

for epoch in range(config["fp32_epochs"]):
    # Training
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, criterion, device, use_amp=False
    )

    # Validation
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    # Learning rate step
    scheduler.step()

    # Track best
    if val_acc > best_fp32_acc:
        best_fp32_acc = val_acc
        save_model(model, fp32_ckpt_path)
        print(f"  → New best model saved (Acc: {val_acc:.2f}%)")

    print(f"Epoch {epoch+1:2d}/{config['fp32_epochs']} | "
          f"Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

fp32_results["best_accuracy"] = best_fp32_acc
print(f"\n✓ FP32 Training complete. Best validation accuracy: {best_fp32_acc:.2f}%")

In [ ]:
# CELL 5b: FINAL FP32 EVALUATION

# Reload the best checkpoint and run one final, deterministic evaluation pass.
# This gives us a single, well-defined (fp32_model, fp32_acc, fp32_loss) triple
# that every later comparison cell relies on — instead of re-deriving it ad hoc.
model.load_state_dict(torch.load(fp32_ckpt_path, map_location=device))
fp32_model = model
fp32_model.eval()
fp32_loss, fp32_acc = evaluate(fp32_model, val_loader, criterion, device)
print(f"\n✓ Final FP32 evaluation | Loss: {fp32_loss:.4f} | Accuracy: {fp32_acc:.2f}%")

## Section 5: Quantization-Aware Training (QAT)

### QAT Process
1. **Copy trained model**: Deep copy of best FP32 checkpoint
2. **Apply QAT config**: Set quantization configuration to FBGEMM backend
3. **Prepare for QAT**: Insert fake quantization modules into model
4. **Fine-tune**: Train for a few epochs (~5), freezing BN stats and observers
   near the end so the model "settles" into the quantized numerics
5. **Convert to INT8**: Replace fake quant with actual INT8 kernels


In [ ]:
# CELL 6: PREPARE FOR QAT

# Deep copy the best FP32 model (already loaded above)
qat_model = copy.deepcopy(fp32_model).cpu()
qat_model.train()

# Apply QAT configuration
qat_model.qconfig = tq.get_default_qat_qconfig("fbgemm")
tq.prepare_qat(qat_model, inplace=True)

# Move to device
qat_model = qat_model.to(device)

print("✓ QAT model prepared")
print(f"  Quantization config: FBGEMM")
print(f"  Backend: {torch.backends.quantized.engine}")

print("Testing single batch...")
x, y = next(iter(train_loader))
print("Batch OK", x.shape, y.shape)

In [ ]:
# CELL 7: QAT FINE-TUNING

qat_optimizer = optim.AdamW(
    qat_model.parameters(),
    lr=config["qat_lr"],
    weight_decay=config["qat_weight_decay"]
)

qat_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    qat_optimizer,
    T_max=config["qat_epochs"]
)

print("Starting QAT fine-tuning...")
print(
    f"Configuration:\n"
    f"  Epochs: {config['qat_epochs']}\n"
    f"  LR: {config['qat_lr']}\n"
    f"  Weight Decay: {config['qat_weight_decay']}"
)

qat_results = {}
best_qat_acc = 0.0
qat_ckpt_path = os.path.join(config["save_dir"], "tinyhybrid_qat.pth")

for epoch in range(config["qat_epochs"]):
    print(f"\n[QAT] Epoch {epoch+1}/{config['qat_epochs']} starting...")
    start_time = time.time()

    # Freeze BatchNorm running stats and fake-quant observers in the later
    # epochs so the model fine-tunes against stable quantization parameters.
    if epoch >= config["qat_freeze_bn_epoch"]:
        qat_model.apply(torch.ao.quantization.freeze_bn_stats)
    if epoch >= config["qat_disable_observer_epoch"]:
        qat_model.apply(torch.ao.quantization.disable_observer)

    # TRAINING
    train_loss, train_acc = train_epoch(
        qat_model,
        train_loader,
        qat_optimizer,
        criterion,
        device,
        use_amp=False
    )
    print(f"[QAT] Epoch {epoch+1} training done")

    # VALIDATION
    val_loss, val_acc = evaluate(qat_model, val_loader, criterion, device)

    # LR SCHEDULER
    qat_scheduler.step()

    # SAVE BEST MODEL
    if val_acc > best_qat_acc:
        best_qat_acc = val_acc
        save_model(qat_model, qat_ckpt_path)
        print(f"[QAT] New best model saved (Acc: {val_acc:.2f}%)")

    epoch_time = time.time() - start_time
    print(
        f"[QAT] Epoch {epoch+1:2d}/{config['qat_epochs']} | "
        f"Time: {epoch_time:.1f}s | "
        f"Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Acc: {val_acc:.2f}%"
    )

qat_results["best_accuracy"] = best_qat_acc
print(f"\n✓ QAT complete | Best validation accuracy: {best_qat_acc:.2f}%")

## Section 6: Convert to INT8 & Export

In [ ]:
# CELL 8: CONVERT TO INT8 & EXPORT

# Load best QAT model
qat_model.load_state_dict(torch.load(qat_ckpt_path, map_location=device))

# Convert to INT8 (replace fake quant with actual INT8 ops).
# Conversion must happen on CPU with the model in eval mode.
qat_model.cpu()
qat_model.eval()
int8_model = tq.convert(qat_model, inplace=False)

print("✓ Model converted to INT8")

# Export via tracing — quantized models with grouped/depthwise convs can be
# brittle under torch.jit.script, so tracing with a representative input
# is the more reliable export path here.
example_input = torch.randn(1, 3, config["img_size"], config["img_size"])
traced_model = torch.jit.trace(int8_model, example_input)
ts_path = os.path.join(config["save_dir"], "tinyhybrid_int8_ts.pt")
traced_model.save(ts_path)
print(f"✓ TorchScript INT8 model saved: {ts_path}")

# Also save state dict
int8_state_path = os.path.join(config["save_dir"], "tinyhybrid_int8.pth")
torch.save(int8_model.state_dict(), int8_state_path)
print(f"✓ INT8 state dict saved: {int8_state_path}")

## Section 7: Evaluation & Results Analysis

In [ ]:
# CELL 9: INT8 MODEL EVALUATION

# The INT8 model only runs on CPU, so evaluation is done there explicitly.
# Reusing the shared evaluate() utility keeps this consistent with how
# fp32_acc/fp32_loss were computed above.
int8_model.eval()
qat_loss, qat_acc = evaluate(int8_model, val_loader, criterion, torch.device("cpu"))

print("\nINT8 Model:")
print(f"  Accuracy: {qat_acc:.2f}%")
print(f"  Loss: {qat_loss:.4f}")

In [ ]:
# CELL 9b: SIZE COMPARISON & QUANTIZATION METRICS

fp32_model_size = os.path.getsize(fp32_ckpt_path) / (1024 ** 2)
int8_model_size = os.path.getsize(int8_state_path) / (1024 ** 2)
ts_size = os.path.getsize(ts_path) / (1024 ** 2)

accuracy_drop = fp32_acc - qat_acc
compression_ratio = fp32_model_size / int8_model_size

print("\n" + "=" * 70)
print("MODEL SIZE COMPARISON")
print("=" * 70)

print(f"\nParameter Count: {count_parameters(fp32_model) / 1e6:.2f}M")

print(f"\nFP32 Size: {fp32_model_size:.2f} MB")
print(f"INT8 Size (quantized): {int8_model_size:.2f} MB")
print(f"TorchScript Size: {ts_size:.2f} MB")

print(f"\nCompression Ratio: {compression_ratio:.2f}× smaller")

print("\n" + "=" * 70)
print("QUANTIZATION IMPACT")
print("=" * 70)

print(
    f"Accuracy Drop (QAT vs FP32): "
    f"{accuracy_drop:.2f}% "
    f"(FP32: {fp32_acc:.2f}% → QAT: {qat_acc:.2f}%)"
)

print(
    f"Size Reduction (INT8): "
    f"{100 * (1 - int8_model_size / fp32_model_size):.1f}%"
)

In [ ]:
# CELL 10: SAVE EXPERIMENT SUMMARY

# Compile comprehensive results
final_results = {
    "fp32": {
        "accuracy": float(fp32_acc),
        "loss": float(fp32_loss),
        "size_mb": float(fp32_model_size),
        "num_parameters": int(count_parameters(fp32_model)),
    },
    "qat_int8": {
        "accuracy": float(qat_acc),
        "loss": float(qat_loss),
        "size_mb": float(int8_model_size),
        "torchscript_size_mb": float(ts_size),
        "note": "Fully converted INT8 model (post tq.convert), not the simulated QAT model"
    },
    "quantization_metrics": {
        "accuracy_drop": float(accuracy_drop),
        "accuracy_drop_percent": float(100 * accuracy_drop / fp32_acc),
        "compression_ratio": float(compression_ratio),
        "size_reduction_percent": float(100 * (1 - int8_model_size / fp32_model_size)),
    },
    "training_config": config.to_dict(),
    "system_info": system_info,
}

# Save summary
summary_path = os.path.join(config["save_dir"], "experiment_summary.json")
create_results_summary(fp32_model, final_results, config.to_dict(), system_info, summary_path)

print(f"\n✓ Experiment summary saved: {summary_path}")

# Print final summary
print("\n" + "=" * 70)
print("EXPERIMENT COMPLETE")
print("=" * 70)
print("\nSaved Files:")
print(f"  ✓ FP32 model: {fp32_ckpt_path}")
print(f"  ✓ QAT model: {qat_ckpt_path}")
print(f"  ✓ INT8 state: {int8_state_path}")
print(f"  ✓ TorchScript: {ts_path}")
print(f"  ✓ Summary: {summary_path}")
print(f"  ✓ Config: {os.path.join(config['save_dir'], 'config.json')}")
print("\nKey Results:")
print(f"  • FP32 Accuracy: {fp32_acc:.2f}%")
print(f"  • QAT Accuracy: {qat_acc:.2f}%")
print(f"  • Accuracy Drop: {accuracy_drop:.2f}%")
print(f"  • Size Reduction: {100 * (1 - int8_model_size / fp32_model_size):.1f}%")